## CKA to PESQ

A per-utterance correlation between layer-wise CKA and PESQ improvement is dominated by SNR: both quantities track degradation severity. To isolate any predictive relationship internal to a fixed degradation level, we remove the SNR-dependent mean effect by within-group centering before correlating. Encoder and latent layers correlate near zero. Decoder and refinement layers show increasingly negative centered correlations through the first decoder block, which then saturate.

Runtime: under 30 seconds on any device.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO.name != "SE-Probe" and REPO.parent != REPO:
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from se_probe.device import device_info, get_device
from se_probe.plotting import MODEL_COLORS, apply_paper_rcparams

apply_paper_rcparams()

DEVICE = get_device()
print(device_info(DEVICE))

import pandas as pd

DATA_DIR = REPO / ("results_df" if (REPO / "results_df").exists() else "results_demo")
print(f"Using data from: {DATA_DIR}")

## Load MUSE table and compute centered residuals

PESQ gain = enhanced_pesq − noisy_pesq, then subtract the per-(layer, SNR) cell mean for both CKA and PESQ-gain. The centered quantities isolate any predictive relationship internal to a fixed degradation level.

In [ ]:
muse_path = DATA_DIR / ("snr/cka_snr_muse.parquet" if (DATA_DIR / "snr").exists() else "cka_snr_muse_demo.parquet")
df = pd.read_parquet(muse_path)
df = df[df["layer"].str.endswith(".norm1")].copy()

df["PESQ_gain"] = df["enhanced_pesq"] - df["noisy_pesq"]
for col in ["CKA", "PESQ_gain"]:
    df[col + "_centered"] = df[col] - df.groupby(["layer", "snr"])[col].transform("mean")

block_order = ["encoder_level1", "encoder_level2", "latent", "decoder_level2", "decoder_level1", "mag_refinement"]
norm1_layers = sorted(df["layer"].unique(),
                       key=lambda L: (next((i for i, b in enumerate(block_order) if b in L), len(block_order)), L))
print(f"norm1 layers: {len(norm1_layers)} | rows after filter: {len(df):,}")

## Per-layer correlation

In [ ]:
rows = []
for layer in norm1_layers:
    g = df[df["layer"] == layer]
    rows.append({
        "layer": layer,
        "rho_pesq": g["CKA_centered"].corr(g["PESQ_gain_centered"]),
    })
corr = pd.DataFrame(rows)
corr.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

block_size = max(1, len(norm1_layers) // 6)
tick_labels = ["Enc-L1", "Enc-L2", "Latent", "Dec-L2", "Dec-L1", "Refinement"]
tick_positions = [i * block_size + block_size / 2 - 0.5 for i in range(6)]
x = np.arange(len(norm1_layers))

fig, ax = plt.subplots(figsize=(6.0, 3.2))
ax.plot(x, corr["rho_pesq"].values, marker="o", linewidth=1.7, color=MODEL_COLORS["muse"])
ax.axhline(0, color="red", linestyle="--", alpha=0.6)
ax.set_xlabel("Block")
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=30, ha="center")
for i in range(block_size, len(norm1_layers), block_size):
    ax.axvline(x=i - 0.5, color="k", linestyle=":", linewidth=0.6)
ax.grid(True, alpha=0.2)
ax.set_ylabel(r"$\rho$ (centered CKA, centered PESQ gain)")
ax.set_title("Per-layer CKA-PESQ correlation, SNR partialled out (MUSE)", fontsize=11)
plt.tight_layout()